# Document Analysis Context Engineering Demo

A complete demonstration of context engineering techniques for processing long documents.

**Scenario**: A developer uploads the AcmeCorp API Reference (~2,500 tokens) and asks:
> *"What are the rate limits and how should I handle them in my code?"

The document exceeds our context budget, requiring progressively smarter context strategies.

| Stage | Technique | Problem Solved |
|-------|-----------|----------------|
| 0 | Naive loading | Baseline — document truncated, sections lost |
| 1 | Fixed-size chunking | Fits budget, but breaks across section boundaries |
| 2 | Semantic chunking + position-aware assembly | Preserves sections, Sandwich Pattern applied |
| 3 | Full pipeline (DocumentAnalysisProcessor) | Hierarchical background + best-position assembly |

In [ ]:
import sys
import json
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from shared import count_tokens
from document_analysis import (
    fixed_size_chunk,
    semantic_chunk,
    position_aware_assemble,
    DocumentAnalysisProcessor,
)

print("✓ Imports successful")

## Step 1: Load Document

Load the API Reference document and measure the gap between document size and our context budget.

In [ ]:
with open("sample_docs/api_reference.md", encoding="utf-8") as f:
    document = f.read()

BUDGET = 1_800  # Tight budget — forces all stages to demonstrate real constraints
QUERY = "What are the rate limits and how should I handle them in my code?"
INSTRUCTION = (
    "You are a technical documentation assistant. "
    "Answer the user's question using only the provided context. "
    "Be precise and include specific values (numbers, codes, HTTP headers)."
)

doc_tokens   = count_tokens(document)
query_tokens = count_tokens(QUERY)
instr_tokens = count_tokens(INSTRUCTION)
overhead     = query_tokens + instr_tokens

print(f"Document:             {doc_tokens:>5} tokens")
print(f"Instruction:          {instr_tokens:>5} tokens")
print(f"Query:                {query_tokens:>5} tokens")
print(f"─" * 35)
print(f"Context budget:       {BUDGET:>5} tokens")
print(f"Available for chunks: {BUDGET - overhead:>5} tokens")

if doc_tokens > BUDGET:
    print(f"\n⚠️  Document exceeds budget by {doc_tokens - BUDGET} tokens "
          f"(document is {doc_tokens / BUDGET:.1%} of budget)")

## Stage 0: Naive Loading

Load the entire document and truncate at the token limit. Fast, but loses arbitrarily.

In [ ]:
from tiktoken import get_encoding
_enc = get_encoding("cl100k_base")

full_context = f"{INSTRUCTION}\n\n{document}\n\n{QUERY}"
full_tokens  = count_tokens(full_context)

if full_tokens > BUDGET:
    # Hard truncation to fit budget
    token_ids = _enc.encode(full_context)[:BUDGET]
    naive_ctx = _enc.decode(token_ids) + "\n[... TRUNCATED ...]"
    naive_tokens = BUDGET
else:
    naive_ctx    = full_context
    naive_tokens = full_tokens

# Identify which ## sections survived truncation
doc_sections     = [l.lstrip("# ").strip() for l in document.splitlines() if l.startswith("## ")]
surviving        = [s for s in doc_sections if s in naive_ctx]
lost             = [s for s in doc_sections if s not in naive_ctx]

print(f"Stage 0: Naive Loading")
print(f"  Assembled:         {naive_tokens:>5} / {BUDGET} tokens")
print(f"  Sections present:  {surviving}")
print(f"  ⚠️  Sections lost:  {lost}")
print(f"\nProblem: truncation is arbitrary — the exact sections lost")
print(f"  depend on where the token limit falls, not on relevance.")

## Stage 1: Fixed-Size Chunking

Split the document into equal-sized chunks and select the first K that fit the budget. Predictable but ignores document structure.

In [ ]:
CHUNK_SIZE = 300

chunks_fixed = fixed_size_chunk(document, chunk_size=CHUNK_SIZE, overlap=0)

# Greedily select chunks until budget is full
available = BUDGET - overhead - 60  # ~60 tokens for separators
selected_fixed: list[str] = []
used_tokens = 0
for chunk in chunks_fixed:
    t = count_tokens(chunk)
    if used_tokens + t <= available:
        selected_fixed.append(chunk)
        used_tokens += t
    else:
        break

context_fixed  = INSTRUCTION + "\n\n" + "\n\n---\n\n".join(selected_fixed) + "\n\n" + QUERY
total_fixed    = count_tokens(context_fixed)

print(f"Stage 1: Fixed-Size Chunking ({CHUNK_SIZE}-token chunks)")
print(f"  Chunks produced:   {len(chunks_fixed)}")
print(f"  Chunks selected:   {len(selected_fixed)} / {len(chunks_fixed)}")
print(f"  Tokens:            {total_fixed:>5} / {BUDGET}")

# Demonstrate the boundary break problem
print(f"\n⚠️  Boundary break between chunk 0 and chunk 1:")
print(f"  ...{chunks_fixed[0][-60:].replace(chr(10), ' ')}")
print(f"  {chunks_fixed[1][:60].replace(chr(10), ' ')}...")
print("\nThe split breaks mid-section — a question about Rate Limits")
print("may land in two different chunks, fragmenting the answer.")

## Stage 2: Semantic Chunking + Position-Aware Assembly

Split at paragraph boundaries to preserve coherence, then assemble using the Sandwich Pattern: instruction at top and bottom, most-relevant chunk just before the query.

In [ ]:
chunks_semantic = semantic_chunk(document, max_tokens=400)

# Heuristic relevance: pick the chunk that contains "Rate Limits" first,
# then fill remaining budget with other chunks.
keyword = "Rate Limits"
relevant_first = [c for c in chunks_semantic if keyword in c]
others         = [c for c in chunks_semantic if keyword not in c]
top_chunks     = (relevant_first + others)[:3]  # top-3

context_semantic, trimmed = position_aware_assemble(
    instruction     = INSTRUCTION,
    background      = "",  # No background summary at this stage
    relevant_chunks = top_chunks,
    query           = QUERY,
)
total_semantic = count_tokens(context_semantic)

print(f"Stage 2: Semantic Chunking + Position-Aware Assembly")
print(f"  Semantic chunks produced: {len(chunks_semantic)}")
print(f"  Chunks selected:          {len(top_chunks)}")
print(f"  Tokens:                   {total_semantic:>5} / {BUDGET}")
print(f"  Trimmed layers:           {trimmed}")
print(f"\n✓ Section boundaries preserved (paragraph-level split)")
print(f"✓ Sandwich Pattern: [INSTRUCTION] at top, [REMINDER] at bottom")
print(f"✓ Most-relevant chunk placed just before [QUERY]")
print(f"\nContext layout preview:")
lines = context_semantic.split("\n")
section_headers = [l for l in lines if l.startswith("[")]
for h in section_headers:
    print(f"  {h}")

## Stage 3: Full Pipeline (DocumentAnalysisProcessor)

The complete pipeline:
1. **Semantic chunking** — paragraph-aware splitting
2. **Hierarchical summarization** — build a summary tree; root becomes the background
3. **Position-aware assembly** — Sandwich Pattern with background + top-K chunks

The background summary gives the model a map of the entire document, even when only a few chunks are in detail.

In [ ]:
processor = DocumentAnalysisProcessor(
    total_budget         = BUDGET * 4,  # Generous budget so trimming doesn't obscure the demo
    chunk_size           = 400,
)

result = processor.process(
    document         = document,
    query            = QUERY,
    instruction      = INSTRUCTION,
    chunk_strategy   = "semantic",
    use_hierarchical = True,
    top_k_chunks     = 3,
)

print(f"Stage 3: DocumentAnalysisProcessor (Full Pipeline)")
print(f"\nPipeline report:")
print(json.dumps(result.summary(), indent=2))

print(f"\nContext layout (structural headers only):")
for line in result.assembled_context.split("\n"):
    if line.startswith("[") or line.startswith("-" * 10):
        print(f"  {line}")

print(f"\n✓ [BACKGROUND] = hierarchical summary tree root ({result.tree_depth} levels)")
print(f"✓ [MOST RELEVANT CHUNK] placed immediately before [QUERY]")
print(f"✓ Model has a document map (background) + detail (top-K chunks)")

In [ ]:
# ── Visual comparison ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Document Analysis: Technique Comparison", fontsize=13, fontweight="bold")

stage_labels  = ["Stage 0\nNaive", "Stage 1\nFixed", "Stage 2\nSemantic", "Stage 3\nFull Pipeline"]
stage_tokens  = [naive_tokens, total_fixed, total_semantic, result.total_tokens]
stage_colors  = ["#F44336", "#FF9800", "#2196F3", "#4CAF50"]

# Left: Token usage vs budget
ax1 = axes[0]
bars = ax1.bar(stage_labels, stage_tokens, color=stage_colors, alpha=0.85, edgecolor="white", width=0.55)
ax1.axhline(BUDGET, color="red", linestyle="--", linewidth=1.8, label=f"Budget ({BUDGET} tokens)")
ax1.set_ylabel("Tokens Used")
ax1.set_title("Token Usage per Stage", fontweight="bold")
ax1.legend()
ax1.set_ylim(0, max(stage_tokens) * 1.25)
for bar, t in zip(bars, stage_tokens):
    ok = "✓" if t <= BUDGET else "✗"
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f"{t}\n{ok}",
        ha="center", va="bottom", fontsize=9, fontweight="bold",
    )

# Right: Qualitative feature matrix
ax2 = axes[1]
ax2.axis("off")
features = [
    ["Feature",                  "S0",  "S1",  "S2",  "S3"],
    ["Within budget",            "✗",   "✓",   "✓",   "✓" ],
    ["Respects section bounds",  "✗",   "✗",   "✓",   "✓" ],
    ["Sandwich Pattern",         "✗",   "✗",   "✓",   "✓" ],
    ["Document-level background","✗",   "✗",   "✗",   "✓" ],
    ["Relevance-ordered chunks", "✗",   "✗",   "partial", "✓"],
]
colors_matrix = [
    ["#ECEFF1"] * 5,
    ["#FAFAFA", "#FFCDD2", "#C8E6C9", "#C8E6C9", "#C8E6C9"],
    ["#FAFAFA", "#FFCDD2", "#FFCDD2", "#C8E6C9", "#C8E6C9"],
    ["#FAFAFA", "#FFCDD2", "#FFCDD2", "#C8E6C9", "#C8E6C9"],
    ["#FAFAFA", "#FFCDD2", "#FFCDD2", "#FFCDD2", "#C8E6C9"],
    ["#FAFAFA", "#FFCDD2", "#FFCDD2", "#FFF9C4", "#C8E6C9"],
]
table = ax2.table(
    cellText=features,
    cellColours=colors_matrix,
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1.2, 1.8)
ax2.set_title("Feature Checklist", fontweight="bold", pad=12)

plt.tight_layout()
plt.savefig("document_analysis_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Chart saved: document_analysis_comparison.png")

## Summary

| Stage | Technique | Within Budget | Section Integrity | Document Awareness |
|-------|-----------|:---:|:---:|:---:|
| 0 | Naive loading | ✗ | ✗ | ✗ |
| 1 | Fixed-size chunking | ✓ | ✗ | ✗ |
| 2 | Semantic + position-aware | ✓ | ✓ | ✗ |
| 3 | **DocumentAnalysisProcessor** | ✓ | ✓ | **✓** |

### Key Insight

Stage 3 adds a **background layer** (the hierarchical summary tree root) that gives the model a map of the entire document before it reads the detail chunks. The model now knows *what exists* even for sections not included in full — critical for questions that span multiple sections or require knowing what's *not* in the retrieved chunks.

### When to Use Each Stage

- **Stage 0**: Only when the document is smaller than the context budget
- **Stage 1**: Rapid prototyping; acceptable when content is uniform (logs, code)
- **Stage 2**: Production QA where section structure matters
- **Stage 3**: Multi-query sessions, legal/medical/technical docs, long-running agents